## 야후 파이넨스 (yfinance)

-  는 저 세계 주식 , ETF, 암호화폐의 실시간 및 과거 데이터를 코드 한 줄로 내 파이썬 노트북에 가뎌와 주는 "무료 금융 데이터 치트키"라이브 러리입니다
-  기존에 사이트에서 CSV파일을 다운로드 할 수 있지만, 종목이 많아지면 관리하기 어렵기 떄문에 라이브러리 사용을 권장합니다

[야후 파이넨스 공식 링크](https://finance.yahoo.com/)

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
from curl_cffi import requests as cffi_requests

warnings.filterwarnings('ignore')

# 윈도우 맑은 고딕 설정
# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 맥 애플고딕 설정
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

In [4]:
session = cffi_requests.Session(impersonate="chrome")

tickers = ["JEPQ", "JEPI", "NVDA", "MSFT"]
all_data = {}

for t in tickers:
    df = yf.download(t, start="2024-01-01", end="2026-09-01", session=session, threads=False)
    all_data[t] = df
    time.sleep(5)

all_data

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


{'JEPQ': Price           Close       High        Low       Open   Volume
 Ticker           JEPQ       JEPQ       JEPQ       JEPQ     JEPQ
 Date                                                           
 2024-01-02  37.038410  37.195320  36.920353  37.195320  3137400
 2024-01-03  36.776905  36.920365  36.732072  36.881512  2316600
 2024-01-04  36.627457  36.874029  36.612512  36.664816  2790100
 2024-01-05  36.724586  36.896441  36.612506  36.642395  2798000
 2024-01-08  37.284996  37.292467  36.806789  36.814263  2124500
 ...               ...        ...        ...        ...      ...
 2026-08-25  59.072330  59.220631  58.849884  59.092104  4602500
 2026-08-26  59.111877  59.210742  58.924030  58.943804  4441800
 2026-08-27  59.625980  59.643774  59.279948  59.448022  4667600
 2026-08-28  59.467796  59.823710  59.388702  59.586434  6185300
 2026-08-31  59.537003  59.566661  59.319497  59.467797  8095700
 
 [668 rows x 5 columns],
 'JEPI': Price           Close       High        Low   

In [5]:
all_data["NVDA"].head()

Price,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,
2024-01-02,48.028793,49.152535,47.457451,49.101684,411254000
2024-01-03,47.431526,48.044747,47.183245,47.347769,320896000
2024-01-04,47.859283,48.359832,47.370698,47.628948,306535000
2024-01-05,48.955105,49.403805,48.166391,48.321942,415039000
2024-01-08,52.101990,52.123929,49.336008,49.368914,642510000


In [6]:
all_data["NVDA"].columns

MultiIndex([( 'Close', 'NVDA'),
            (  'High', 'NVDA'),
            (   'Low', 'NVDA'),
            (  'Open', 'NVDA'),
            ('Volume', 'NVDA')],
           names=['Price', 'Ticker'])

## 데이터 준비


In [22]:
close_df = pd.DataFrame()

for t in tickers:
    closs_df[t] = all_data[t]["Close"]

print(closs_df)

Empty DataFrame
Columns: []
Index: []


## 데이터 전처리 

In [23]:
closs_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [24]:
closs_df.describe()


ValueError: Cannot describe a DataFrame without columns

## 초기 가설 
- NVDA, MSFT 갭별 성장주 이기 때문에 단기 변동성이 크더라도 복리효과가 누적되면서 장기적인 우상향을 보여줄  것이다
- JEPQ, JEPI 배당 성장현 ETF이기때문에 수익룰이 낮고 , 상승기가 낮더라도 시장의 조정기 마다 고점 대비 자산이 깎이는 낙폭으이 낮았을 것이다
- 2024년도 조정기에 ETF 중 표준편차가 낮은 JPI가 방어력아 가장 좋았을 것이다 

In [25]:
# .pct_change() : 뱐화율을 구해주느 ㄴ함수 
# ex)_ (오늘 종가 - 어제 종가) /어제 종가

returns_df = closs_df.pct_change().dropna()
returns_df

,JEPQ,JEPI,NVDA,MSFT
Date,,,,
2024-01-03,-0.007060,-0.005624,-0.012436,-0.000728
2024-01-04,-0.004064,-0.000365,0.009018,-0.007177
2024-01-05,0.002652,-0.000730,0.022897,-0.000516
2024-01-08,0.015260,0.006758,0.064281,0.018872
2024-01-09,0.002204,-0.000726,0.016975,0.002936
...,...,...,...,...
2026-08-25,0.005554,0.000861,0.021921,0.009029
2026-08-26,0.000669,0.000344,-0.015912,0.009477
2026-08-27,0.008697,-0.005330,0.087380,0.017507


In [26]:
# cumprod# 
returns_df = (1 + returns_df).cumprod() - 1
returns_df 


,JEPQ,JEPI,NVDA,MSFT
Date,,,,
2024-01-03,-0.007060,-5.624141e-03,-0.012436,-0.000728
2024-01-04,-0.011095,-5.987151e-03,-0.003529,-0.007900
2024-01-05,-0.008473,-6.712489e-03,0.019287,-0.008413
2024-01-08,0.006658,-1.110223e-16,0.084807,0.010300
2024-01-09,0.008876,-7.255937e-04,0.103222,0.013266
...,...,...,...,...
2026-08-25,0.594894,2.924901e-01,3.430923,0.354133
2026-08-26,0.595962,2.929347e-01,3.360419,0.366966
2026-08-27,0.609842,2.860431e-01,3.741430,0.390898


In [31]:
# cummax: 이전 값과 비효하여 최고 값으로 대체
peak_df = closs_df.cummax()
peak_df

,JEPQ,JEPI,NVDA,MSFT
Date,,,,
2024-01-02,37.038410,44.692699,48.028793,363.117950
2024-01-03,37.038410,44.692699,48.028793,363.117950
2024-01-04,37.038410,44.692699,48.028793,363.117950
2024-01-05,37.038410,44.692699,48.955105,363.117950
2024-01-08,37.284996,44.692699,52.101990,366.858093
...,...,...,...,...
2026-08-25,59.873146,57.764870,235.202393,537.646362
2026-08-26,59.873146,57.784740,235.202393,537.646362
2026-08-27,59.873146,57.784740,235.202393,537.646362


In [32]:
# 고점대비 낙폭 
drawdown_df = (closs_df - peak_df) / peak_df
drawdown_df

,JEPQ,JEPI,NVDA,MSFT
Date,,,,
2024-01-02,0.000000,0.000000,0.000000,0.000000
2024-01-03,-0.007060,-0.005624,-0.012436,-0.000728
2024-01-04,-0.011095,-0.005987,-0.003529,-0.007900
2024-01-05,-0.008473,-0.006712,0.000000,-0.008413
2024-01-08,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...
2026-08-25,-0.013375,0.000000,-0.095197,-0.085440
2026-08-26,-0.012715,0.000000,-0.109594,-0.076772
2026-08-27,-0.004128,-0.005330,-0.031791,-0.060609


In [35]:
# 수익의 변동성 (표준편차) + 미국주식 (252) - 휴장일을 기준으로 미국의 주식 일수 
# 연율화 (미국의 주식의 "년"단위 변동성)

annual_vol = returns_df.std() * np.sqrt(252)
annual_vol

mdd = drawdown_df.min()
mdd

v1_proof == pd.DataFrame({
    "최종 누적 수익률(%)": cum_return_df.iloc[-1] * 100, 
    "연율화 변동성 (%)": annual_vol * 100, 
    "최대 낙폭 MDD(%)": mdd * 100 
})
v1_proof

NameError: name 'v1_proof' is not defined

In [36]:
cum_return_df.index()


NameError: name 'cum_return_df' is not defined

In [ ]:
plt.subplots(2, 1, figsize= (14, 12), sharex=True)

for i in tickers:
    axes[0].plot(cum_return_df.index, cum_return_df[t] * 100, label=t)
    axes[0].set_title("자산별 누적 수익률(2024-2026)")
    axes[0].set_ylabel("누적 수익율")
    axes[0].legend()
    # alpha=0.3: 투명도  30%
    axes[0].grid(True, alpha=0.3)

for i in tickers:
    axes[1].plot(drawdown_df.index, drawdown_df[t] * 100, label=t)
    axes[1].set_title("자산별 누적 수익률(2024-2026)")
    axes[1].set_ylabel("누적 수익율")
    axes[1].legend()

    axes[1].grid(True, alpha=0.3)

## 가설 2
- 배당형 ETP
- JEPQ는 나스갇 100 기반으로 상장된 ETF이므로, NVDA , MSFT와 높은 상관간개를 를 가질 것이다
- JEPI는 S&P 500 기반으로 산자ㅇ도ㅟㄴ EfF이므로,  NVDA , MSFT낮은 상관관계를 가질 것이다 

In [ ]:
# .rolling(): 20일 설정 
# .corr)_: 최근 20일 간의 상관관계를 매일 계산해서 반환

rolling_jepi = return_df["NVDA".rolllomg(window=20).corr(retrns_df["JEPI"])]
rolling_jepi = return_df["NVDA".rolllomg(window=20).corr(retrns_df["JEPI"])]

In [37]:
plt.figure(figsize=(14, 6))
plt.plot(rolling_jpei.index, rolling_jpei, label="NVDA와 JEPI의 상관관계", color="orange")
plt.plot(rolling_jpeq.index, rolling_jpeq, label="NVDA와 JEPQ의 상관관계", color="blue")

plt.axhline(0.7, color="red", linestyle="--", alpha=0.3, label="강한 상관 관계")
plt.axhline(0.3, color="green", linestyle="--", alpha=0.3, label="약한 상관 관계")

plt.title("시간의 흐름에 따른 NVDA 대비 JEPI vs JEPQ의 상관계수 추이(20일)")
plt.ylabel("상관 계수")
plt.xlabel("일자")
plt.legend(loc="lower left")
plt.grid(True, alpha=0.2)
plt.show()

NameError: name 'rolling_jpei' is not defined

<Figure size 1400x600 with 0 Axes>

## 가설 검증 결과 
- JEPQ와 NVDA는 지속적 0.7이상의 높은 상관관계를 유지하며 동조화가 잘 되고 있다
- 반대로 는  와의 상관관계가 0.4 의 낮은 상관관계를 유지하며 동조화가 잘 되고 있다
- 성장두 리스크를 부나산하고 싶을 때는 JEPI를 추천해서 자산을 배분해야한다
- 가설은 찬, 가설 채택

## EDA 시각화 
역사적 고점 대비 대략 20%가 떨어질 때 사면, 이후 한 달 뒤에 얼마정도릐 수익을 볼 확률, 평균 수익룰은 어떻게될까?

In [38]:
forward_days = 20
result = []

for i in tickers:
    # 20일 뒤 주가 / 현재 주가 (20일 종안의 수익율)
    rwd_return_df = (close_df[t].shift(-forward_days) / close_df[t])\

    drawdown = drawdown_df_df[t]
    df_temp = pd.DataFrame({"Drawdown": drawdown, "FwdReturn": fwd_return_df}).dropna()

    # 구간
    bins = [-1.0, -0.2, -0.1, -0.05, 0.001]
    labels = ["-20%이하(대폭락)", "-10% ~ -20%", "-5% ~ -10%", "0% ~ -5%"]

    df_temp["구간"] = pd.cut(df_temp["Drawdown"], bins=bins, labels= labels

print(df_temp)

KeyError: 'MSFT'